# 부품 자동추천 AI — Colab 파이프라인 (수집 + QLoRA 학습)

Google Colab **Pro (A100 40GB 권장)** 에서 데이터 수집부터 Qwen3-14B QLoRA 학습·추론까지 한 번에 실행합니다.

1. GPU/환경 확인 → 2. Google Drive 마운트 → 3. 코드/의존성 설치 → 4. (수집용) llama.cpp 서버 기동 →
5. 시크릿 설정 → 6. 데이터 수집 → 7. 전처리 → 8. QLoRA 학습 → 9. 추론 테스트

> **런타임 → 런타임 유형 변경**에서 A100을 선택했는지 먼저 확인하세요. Colab 런타임은 휘발성이므로 데이터/어댑터는 모두 Google Drive에 저장합니다.
>
> ⚠️ **순서 중요**: 6번(수집)은 4번 llama.cpp 서버가 `✅ 준비됨` 이 뜬 뒤에 실행하세요. 서버가 없으면 `Connection error` 가 납니다.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. Google Drive 마운트
수집 데이터·학습 산출물(어댑터)을 Drive에 영구 저장합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/auto_search'   # Drive 내 저장 위치
os.makedirs(f'{DRIVE_ROOT}/datasheets', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/models', exist_ok=True)
print('Drive 저장 경로:', DRIVE_ROOT)

## 3. 코드 가져오기 + 의존성 설치
아래 `REPO_URL`을 본인 저장소 주소로 바꾸세요. (또는 Drive에 올려둔 코드를 사용)

In [ ]:
REPO_URL = 'https://github.com/Merlion1346/Auto_Part_Search.git'
%cd /content
![ -d Auto_Part_Search ] || git clone $REPO_URL
%cd /content/Auto_Part_Search

# 데이터시트 PDF 파싱용 Java (OpenDataLoader 는 JVM 기반)
!apt-get -qq install -y default-jre > /dev/null && java -version

# Python 의존성: Colab 기본 torch/transformers는 그대로 두고 부족한 것만 설치/갱신
# (requirements.txt 는 torch를 핀하지 않고 최소 버전만 지정)
!pip install -q -U -r requirements.txt

## 4. (데이터 수집용) llama.cpp 서버 기동
수집 단계의 사양 추출/요구사항 증강은 OpenAI 호환 LLM 엔드포인트를 사용합니다.
llama.cpp 를 CUDA로 빌드한 뒤 `llama-server` 로 Qwen3-14B(GGUF)를 OpenAI 호환(`/v1`)으로 서빙합니다.

> **GGUF는 Drive(`models/gguf/`)에 저장**되어, 다음 세션부터는 재다운로드 없이 바로 로드합니다(첫 실행만 ~9GB 다운로드 + Drive 복사).
>
> Qwen3의 reasoning(thinking)은 `--reasoning-budget 0` 으로 꺼서 추출 속도·안정성을 높입니다. `response_format=json_object` 가 grammar 로 JSON 출력을 강제하므로 파싱도 안전합니다.
>
> 이미 만들어 둔 카탈로그(`parts_catalog.json`)를 Drive에서 재사용한다면 이 셀과 6번(수집)은 건너뛰어도 됩니다. 외부 API(DashScope 등)를 쓸 경우엔 5번에서 `LLM_*`만 바꾸고 이 셀을 건너뜁니다.

In [ ]:
# llama.cpp 빌드 (CUDA)
!apt-get -qq install -y libcurl4-openssl-dev > /dev/null
![ -d /content/llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=ON
!cmake --build /content/llama.cpp/build --config Release -j --target llama-server

# GGUF 확보: Drive에 있으면 재사용(재다운로드 X), 없으면 받아서 Drive에 저장
import os, shutil, subprocess, time, urllib.request
from huggingface_hub import HfApi, hf_hub_download

REPO, QUANT = "Qwen/Qwen3-14B-GGUF", "Q4_K_M"
GGUF_DIR = f"{DRIVE_ROOT}/models/gguf"      # Drive 영구 저장 위치
os.makedirs(GGUF_DIR, exist_ok=True)

cached = [f for f in os.listdir(GGUF_DIR)
          if f.lower().endswith(".gguf") and QUANT.lower() in f.lower()]
if cached:
    GGUF = os.path.join(GGUF_DIR, cached[0])   # 다음 세션: Drive에서 바로 로드
    print("Drive 캐시 사용 (다운로드 생략):", GGUF)
else:
    cands = [f for f in HfApi().list_repo_files(REPO)
             if f.lower().endswith(".gguf") and QUANT.lower() in f.lower()]
    assert cands, f"{REPO} 에서 {QUANT} GGUF 를 찾지 못함"
    print("GGUF 다운로드:", cands[0])
    tmp = hf_hub_download(REPO, cands[0])       # 빠른 로컬 캐시로 다운로드(진행바)
    dst = os.path.join(GGUF_DIR, os.path.basename(cands[0]))
    print("Drive로 복사 중...(약 9GB, 수 분 소요 — 다음 세션부터 재사용)")
    shutil.copy(tmp, dst)
    print("Drive 저장 완료:", dst)
    GGUF = tmp                                  # 이번 세션은 빠른 로컬에서 로드

# OpenAI 호환 서버 기동 (--reasoning-budget 0: Qwen3 thinking off)
LOG = "/content/llama_server.log"
_tail = lambda n=40: subprocess.run(["tail", "-n", str(n), LOG], capture_output=True, text=True).stdout
subprocess.run(["pkill", "-f", "llama-server"])  # 이전 서버 정리
time.sleep(2)
logf = open(LOG, "w")
server = subprocess.Popen([
    "/content/llama.cpp/build/bin/llama-server",
    "-m", GGUF,
    "--host", "127.0.0.1", "--port", "8000",
    "-ngl", "999", "-c", "8192",
    "--jinja", "--reasoning-budget", "0",
], stdout=logf, stderr=subprocess.STDOUT)

ready = False
for _ in range(180):  # Drive 로드는 디스크보다 조금 느릴 수 있음
    if server.poll() is not None:
        print(f"❌ llama-server 종료 (code={server.returncode}). 로그 마지막:\n"); print(_tail()); break
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("✅ llama-server 준비됨 (http://localhost:8000/v1)"); ready = True; break
    except Exception:
        time.sleep(2)
if not ready and server.poll() is None:
    print("⏳ 아직 응답 없음. 현재 로그:\n"); print(_tail(20))

## 5. 시크릿 / .env 설정
Colab 좌측 🔑(보안 비밀)에 `MOUSER_API_KEY`(또는 Digi-Key 키)를 등록한 뒤 실행하세요.
경로는 Drive로 지정해 런타임 재시작에도 보존되게 합니다.

In [ ]:
%cd /content/Auto_Part_Search
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ''
    except Exception:
        return ''

env = f"""
MOUSER_API_KEY={secret('MOUSER_API_KEY')}
MOUSER_BASE=https://api.mouser.com
DIGIKEY_CLIENT_ID={secret('DIGIKEY_CLIENT_ID')}
DIGIKEY_CLIENT_SECRET={secret('DIGIKEY_CLIENT_SECRET')}
DIGIKEY_BASE=https://api.digikey.com

LLM_BASE_URL=http://localhost:8000/v1
LLM_API_KEY=EMPTY
LLM_MODEL=qwen3-14b

PDF_DIR={DRIVE_ROOT}/datasheets
CATALOG_PATH={DRIVE_ROOT}/parts_catalog.json
"""
with open('.env', 'w') as f:
    f.write(env)
print('.env 작성 완료 (cwd:', __import__('os').getcwd(), ')')

## 6. 데이터 수집 (유통사 API + 데이터시트 파싱 + LLM 사양 추출)
→ `parts_catalog.json` 생성(Drive에 저장)

In [ ]:
%cd /content/Auto_Part_Search
!python -m pipeline.build_catalog --source mouser \
    --keywords-file pipeline/keywords.example.txt --limit 15

## 7. 학습 데이터 전처리
카탈로그 → TRL 대화 데이터셋(`train.jsonl`). `--augment` 는 llama.cpp 서버로 요구사항을 다양화합니다(4번 서버 필요).

In [ ]:
%cd /content/Auto_Part_Search
!python train/preprocess.py \
    --input-glob "$DRIVE_ROOT/parts_catalog.json" \
    --output "$DRIVE_ROOT/train.jsonl" \
    --augment --num-variations 5

## 8. QLoRA 학습 (Qwen3-14B)
어댑터는 Drive에 저장됩니다. A100 40GB 기준 기본값입니다. VRAM이 작은 GPU(L4/T4)면 `--max-seq-len 1024 --batch-size 1` 로 낮추세요.

In [ ]:
%cd /content/Auto_Part_Search
!python train/train_qlora.py \
    --train-file "$DRIVE_ROOT/train.jsonl" \
    --output-dir "$DRIVE_ROOT/models/qwen3-14b-parts-lora"

## 9. 추론 테스트

In [ ]:
%cd /content/Auto_Part_Search
!python src/recommend.py \
    --adapter "$DRIVE_ROOT/models/qwen3-14b-parts-lora" \
    "5V를 3.3V로 변환하고 1A 이상 출력하는 LDO 추천해줘"